Para abrir o notebook no Google Colab, altere o domínio `github.com` para `githubtocolab.com`

# PSI5892 - Aula de Exercícios

# Medidas de desempenho

Neste exercício vamos avaliar um modelo treinado para o problema de classificação do banco de dados [Fashion MNIST](https://arxiv.org/abs/1708.07747), usando o conjunto de dados de teste com 10000 imagens.

Vamos utilizar um modelo pré-treinado com os pesos salvos no arquivo [model.pt](https://github.com/psi5892/exercicios_aula_publico/raw/refs/heads/main/ex_medidas/model.pt).

Para instanciar o modelo, primeiramente é necessário definir a arquitetura e, em seguida, carregar os pesos, o que pode ser feito com o código:

``` python
import numpy as np
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, input_size=28*28, n_hidden=20, output_size=10):
        # Necessário chamar __init__() da classe mãe
        super().__init__()
        self.input_size = input_size
        self.network = nn.Sequential(
            nn.Linear(input_size, n_hidden), 
            nn.ReLU(), 
            nn.Linear(n_hidden, n_hidden), 
            nn.ReLU(), 
            nn.Linear(n_hidden, output_size), 
            #nn.Softmax()
        )
        
    def forward(self, x):
        x = x.view(-1, self.input_size)
        return self.network(x)

model = Model()
model.eval()
model.load_state_dict(torch.load("model.pt"))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
```

O *data loader* com os dados pode ser criado utilizando a biblioteca `torchvision`, como fizemos no exercício para o treinamento da rede:

``` python
from torchvision import datasets, transforms

dir_data = "~/temp"
Nb_test = 10000

test_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        dir_data,
        train=False,
        transform=transforms.Compose(            
            [transforms.ToTensor()]
        ),
    ),
    batch_size=Nb_test,
    shuffle=True,
)
```

Esse modelo busca resolver um problema de classificação multiclasse com 10 clases. Para avaliar o seu desempenho, vamos utilizar métricas de classificação binária. Para cada uma das 10 classes, vamos avaliar o desempenho em termos da classificação para a classe em questão vs. a classificação em qualquer uma das outras classes.

# Exercício 1

Para cada modelo de classificação binária que classifica uma classe X vs. qualquer das outras nove classes, apresente as seguintes métricas:
- A matriz de confusão;
- A acurácia;
- A precisão;
- A revocação;
- O *F1 Score*;
- O Coeficiente de correlação de Matthew;
- A curva ROC;
- O valor da AUC.

Dicas:
- O resultado para um modelo de classificação binária de uma classe X vs. qualquer das outras nove classes pode ser obtido a partir da saída da camada de *Softmax* para a classe X. Basta comparar a saída com um limiar, que é tipicamente igual a 0,5. Se a saída for maior que o limiar, consideramos que a entrada foi classificada na classe X, caso contrário, não;
- Note que o modelo carregado não inclui a camada `Softmax`, pois como mencionado no exercício para treinamento do modelo, a função custo `CrossEntropyLoss`espera receber os valores dos *logits*. Dessa forma, para obter os valores após a camada *Softmax*, utilize a função `F.softmax` após o cálculo da saída do modelo:

``` python
import torch.nn.functional as F

# Calcula a saída do modelo
y = model(X)

# Aplica Softmax
y2 = F.softmax(y, dim=1)
```
- Para o cálculo da matriz de confusão e as métricas derivadas dela, é necessário calcular o valor da saída binária após a decisão aplicando um limiar (tipicamente 0,5);
- Para o cálculo da ROC e AUC, é necessário o valor da probabilidade, antes da decisão;
- As métricas podem ser calculadas com a biblioteca `sklearn`. Seguem os links para a documentação das funções:
  - [Matriz de confusão](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.confusion_matrix.html);
    - Note que a matrix de confusão do `sklearn` tem um formato diferente do usual, apresentado na aula.
  - [Acurácia](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.accuracy_score.html);
  - [Precisão](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.precision_score.html);
  - [Revocação](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.recall_score.html);
  - [*F1 Score*](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.f1_score.html);
  - [Coeficiente de correlação de Matthew](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.matthews_corrcoef.html);
  - [Curva ROC](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.roc_curve.html);
  - [Valor da AUC](https://scikit-learn.org/dev/modules/generated/sklearn.metrics.auc.html).
- Para plotar a curva ROC e calcular a AUC, pode-se utilizar o código:
``` python
from sklearn.metrics import roc_curve, auc

roc = roc_curve(real_labels, model_output)
auc = auc(roc[0], roc[1])

plt.figure()
plt.plot(roc[0], roc[1], label='Model')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Random guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
_ = plt.legend(loc='lower right')

print(f'AUC: {auc}')

```


## Resolução

# Exercício 2

Responda:
- Qual é o melhor classificador em termos de AUC? E o pior?
- O que poderia ser feito para aumentar a precisão do pior classificador?